<a href="https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: ML Appendix — "What Predicts Health?" (Random Forest feature importance for health score)

Methodology question: Health Score is explicitly defined in the paper as Impressions (30pts) +
Position (30pts) + CTR (20pts) + Scroll Depth (20pts). The Random Forest then finds Average
Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of... health
score. But those are literally three of the four ingredients used to compute health score in
the first place. The paper does flag this ("importance is descriptive rather than causal"), which
is honest of them — but I'd still ask: if 90% of the model's importance comes from components
of its own label, what is this model actually telling us that the health-score formula doesn't
already say directly? Is there a version of this analysis using a label that isn't
partially self-referential?

Finding 2: Finding #10 — "AI Model Performance" (OpenAI vs. Gemini, age-controlled cohorts)

Methodology question: The paper controls for content age when comparing OpenAI vs. Gemini
health scores, which is a real and good step — most naive comparisons wouldn't even do that.
My question is about what's still uncontrolled: were pages assigned to a model provider
randomly, or did certain clients, topics, or editors consistently prefer one provider over the
other? If e.g. one client with unusually strong topical authority happens to use Gemini more,
age-controlling wouldn't catch that, and the "Gemini leads some cohorts" result could still be
picking up client or topic effects rather than a genuine model-quality difference. Is there a
client-controlled or topic-controlled cut of this same comparison?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used a client-grouped split, which is the honest design. To show the
before/after clearly, I'm re-running the same model with a naive random row split as the
"before" (the common mistake), compared against my actual grouped split as the "after" — same
data, same label, same model, only the split logic changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/ArishaRamzan-dev/arisha-flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['underperforming'] = (df['trend_pct'] < 0).astype(int)

leak_cols = ['trend_pct', 'trend_direction', 'impressions_last_30d', 'clicks_last_30d',
             'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
             'content_id', 'client_id', 'underperforming', 'action_score', 'reason_code', 'action_label',
             'staleness_bucket', 'position_bucket']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in leak_cols]

def precision_at_k(y_true, y_scores, k=50):
    top_k_idx = np.argsort(y_scores)[-k:]
    return y_true.iloc[top_k_idx].mean()

# BEFORE: naive random split (the common mistake — no grouping)
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    df[feature_cols].fillna(0), df['underperforming'], test_size=0.2, random_state=42)
rf_bad = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced").fit(X_tr_bad, y_tr_bad)
score_bad = precision_at_k(y_te_bad, rf_bad.predict_proba(X_te_bad)[:, 1], 50)

# AFTER: honest client-grouped split (my actual Week-5 design)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
X_tr_good, X_te_good = df.loc[train_idx, feature_cols].fillna(0), df.loc[test_idx, feature_cols].fillna(0)
y_tr_good, y_te_good = df.loc[train_idx, 'underperforming'], df.loc[test_idx, 'underperforming']
rf_good = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced").fit(X_tr_good, y_tr_good)
score_good = precision_at_k(y_te_good, rf_good.predict_proba(X_te_good)[:, 1], 50)

comparison = pd.DataFrame({
    "Split": ["BEFORE — naive random split", "AFTER — client-grouped split (honest)"],
    "Precision@50": [score_bad, score_good]
})
print(comparison.to_string(index=False))

train_clients = set(df.loc[train_idx, 'client_id']) if 'client_id' in df.columns else set()

                                Split  Precision@50
          BEFORE — naive random split          0.92
AFTER — client-grouped split (honest)          0.76


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit on my Week-5 feature set: checking that no feature used to predict "underperforming"
is itself derived from the label, from a future window relative to the label, or from a column the
baseline rule already consumed in a way that would let the model "cheat."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check: does any feature correlate suspiciously highly with the label? (possible leakage signal)
correlations = df[feature_cols + ['underperforming']].corr()['underperforming'].drop('underperforming').sort_values(key=abs, ascending=False)
print("Top 5 feature correlations with label (checking for suspiciously high values):")
print(correlations.head(5))

print("\nExcluded as leakage risks:", leak_cols)
print("""
Reasoning: trend_pct/trend_direction directly define the label, so they're excluded outright.
impressions/clicks/sessions_last_30d and _prev_30d are the raw components trend_pct is computed
from, so including them would let the model reconstruct the label almost exactly — excluded.
action_score/reason_code/action_label are the baseline rule's own outputs, not real-world
features, and including them would mean 'predicting' something we already show as the baseline
comparison — excluded. content_id and client_id are identifiers, not signal — excluded.
""")

Top 5 feature correlations with label (checking for suspiciously high values):
days_with_impressions     0.343823
word_count                0.151164
char_count                0.124376
days_since_last_update    0.121877
days_with_sessions        0.104120
Name: underperforming, dtype: float64

Excluded as leakage risks: ['trend_pct', 'trend_direction', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_id', 'client_id', 'underperforming', 'action_score', 'reason_code', 'action_label', 'staleness_bucket', 'position_bucket']

Reasoning: trend_pct/trend_direction directly define the label, so they're excluded outright.
impressions/clicks/sessions_last_30d and _prev_30d are the raw components trend_pct is computed
from, so including them would let the model reconstruct the label almost exactly — excluded.
action_score/reason_code/action_label are the baseline rule's own outputs, not real-world
features, and

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim (from my Week-5 notebook): "The model clearly beats the hand-written baseline
rule." Rewritten with safe language: "In this client-grouped test split, Logistic Regression
(Precision@50 = 0.88) and Random Forest (0.76) both measured higher than the baseline rule
(0.60) on this specific dataset and split. This is a directional, decision-support signal for
this dataset, not proof the model generalizes to other clients' content beyond what's in this
sample, since Random Forest still misclassifies 39.4% of test rows overall, and Logistic
Regression unexpectedly outperformed the more complex model — a result that should be checked
again given ordinary run-to-run variance before treating it as a stable ranking of methods."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.